# 🌦️ Weather Data Analysis and Prediction

**Domain:** Data Analysis & Visualization  
**Tools:** Python, pandas, NumPy, Matplotlib, Seaborn  
**Author:** Jainish Dabgar, Vraj Padhiyar  
**Roll No:** 12402120601033,12402120601085  
**Subject:** Python Programming  

---

## 📌 Project Objective

Analyze historical weather data and create predictions for **temperature**, **humidity**, and **rainfall** patterns using:
- Statistical analysis & trend identification
- Seasonal pattern visualization
- Simple forecasting using Moving Averages

---

## 📋 Table of Contents
1. [Import Libraries](#1)
2. [Load Dataset](#2)
3. [Data Cleaning & Preprocessing](#3)
4. [Exploratory Data Analysis (EDA)](#4)
5. [Statistical Analysis](#5)
6. [Seasonal Pattern Analysis](#6)
7. [Forecasting with Moving Averages](#7)
8. [Visualization Dashboard](#8)
9. [Conclusion](#9)

---
## 1. Import Libraries <a id='1'></a>

We import all required libraries at the top — this follows PEP 8 standards and keeps the notebook organized.

In [ ]:
# Standard library
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Settings
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ All libraries imported successfully!')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')

---
## 2. Load Dataset <a id='2'></a>

We use a **synthetic dataset** simulating Indian climate patterns (2021–2023).

> 💡 **Alternative:** You can use real data from [Kaggle](https://www.kaggle.com/datasets/muthuj7/weather-dataset) or the OpenWeatherMap API. Just replace the CSV path below.

In [ ]:
# ── Generate synthetic dataset (run once) ───────────────────────────────
np.random.seed(42)

dates = pd.date_range(start='2021-01-01', end='2023-12-31', freq='D')
n = len(dates)
day_of_year = np.array([d.timetuple().tm_yday for d in dates])
month_arr   = np.array([d.month for d in dates])

# Temperature: seasonal sine wave + noise (Indian climate)
temperature = (
    25 + 10 * np.sin(2 * np.pi * (day_of_year - 80) / 365)
    + np.random.normal(0, 2, n)
)

# Humidity: peaks in monsoon (June–Sep)
humidity = (
    60 + 20 * np.sin(2 * np.pi * (day_of_year - 160) / 365)
    + np.random.normal(0, 5, n)
)
humidity = np.clip(humidity, 20, 100)

# Rainfall: mostly in monsoon months
rainfall_prob = np.where((month_arr >= 6) & (month_arr <= 9), 0.5, 0.1)
rainfall = np.where(
    np.random.rand(n) < rainfall_prob,
    np.random.exponential(5, n), 0.0
)

# Create DataFrame
df_raw = pd.DataFrame({
    'date':        dates,
    'temperature': np.round(temperature, 2),
    'humidity':    np.round(humidity, 2),
    'rainfall':    np.round(rainfall, 2)
})

# Save to CSV
os.makedirs('../data', exist_ok=True)
df_raw.to_csv('../data/weather_data.csv', index=False)
print(f'✅ Dataset created: {len(df_raw)} records | 2021-01-01 to 2023-12-31')
print(f'   Columns: {list(df_raw.columns)}')

In [ ]:
# ── Load from CSV ────────────────────────────────────────────────────────
df = pd.read_csv('../data/weather_data.csv', parse_dates=['date'])

print(f'Shape : {df.shape}  (rows, columns)')
print(f'Range : {df["date"].min().date()} → {df["date"].max().date()}')
print()
df.head(10)

In [ ]:
# Dataset info — data types and non-null counts
df.info()

---
## 3. Data Cleaning & Preprocessing <a id='3'></a>

Before analysis, we must:
- Check for **missing values**
- Remove **outliers** (temperature beyond 3σ)
- **Clip** invalid ranges (humidity > 100, negative rainfall)
- Extract **time features** (month, year, season)

In [ ]:
# ── Check missing values ─────────────────────────────────────────────────
print('Missing Values per Column:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
# ── Data cleaning ────────────────────────────────────────────────────────
df = df.sort_values('date').reset_index(drop=True)

# Handle missing values (forward fill then mean)
numeric_cols = ['temperature', 'humidity', 'rainfall']
df[numeric_cols] = df[numeric_cols].fillna(method='ffill').fillna(df[numeric_cols].mean())

# Remove temperature outliers (outside 3 standard deviations)
temp_mean = df['temperature'].mean()
temp_std  = df['temperature'].std()
before    = len(df)
df = df[(df['temperature'] >= temp_mean - 3*temp_std) &
        (df['temperature'] <= temp_mean + 3*temp_std)]
print(f'Outliers removed: {before - len(df)} rows')

# Clip to valid ranges
df['humidity'] = df['humidity'].clip(0, 100)
df['rainfall'] = df['rainfall'].clip(lower=0)

df = df.reset_index(drop=True)
print(f'✅ Cleaned dataset: {len(df)} records')

In [ ]:
# ── Extract time features ────────────────────────────────────────────────
df['year']       = df['date'].dt.year
df['month']      = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%b')
df['day_of_year']= df['date'].dt.dayofyear

def get_season(month):
    """Map month number to Indian meteorological season."""
    if month in [12, 1, 2]:  return 'Winter'
    elif month in [3, 4, 5]: return 'Summer'
    elif month in [6,7,8,9]: return 'Monsoon'
    else:                    return 'Post-Monsoon'

df['season'] = df['month'].apply(get_season)

print('✅ Time features extracted!')
print(f'   Columns added: year, month, month_name, day_of_year, season')
df[['date','temperature','humidity','rainfall','season']].head(8)

---
## 4. Exploratory Data Analysis (EDA) <a id='4'></a>

EDA helps us understand the data shape, distributions, and relationships before building any models.

In [ ]:
# ── Descriptive statistics ───────────────────────────────────────────────
print('📊 Summary Statistics:')
df[numeric_cols].describe().round(2)

In [ ]:
# ── Distribution plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Distribution of Weather Variables', fontsize=14, fontweight='bold')

colors = ['steelblue', 'seagreen', 'darkorange']
labels = ['Temperature (°C)', 'Humidity (%)', 'Rainfall (mm)']

for ax, col, color, label in zip(axes, numeric_cols, colors, labels):
    ax.hist(df[col], bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.1f}')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/distributions.png')
plt.show()
print('💡 Observations:')
print('   - Temperature follows a bimodal/seasonal distribution (hot summers + cool winters)')
print('   - Rainfall is heavily right-skewed — most days have 0mm (dry days)')

In [ ]:
# ── Correlation heatmap ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            linewidths=0.5, ax=ax, square=True, vmin=-1, vmax=1)
ax.set_title('Pearson Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/correlation_heatmap.png')
plt.show()

print('\n📊 Correlation Matrix:')
print(corr.round(3))
print('\n💡 Insight: All variables show weak-to-moderate positive correlation.')
print('   During monsoon: high humidity AND high rainfall tend to occur together.')

In [ ]:
# ── Scatter plot matrix (pairplot) ───────────────────────────────────────
plot_df = df[numeric_cols + ['season']].copy()
season_order = ['Winter', 'Summer', 'Monsoon', 'Post-Monsoon']

g = sns.pairplot(plot_df, hue='season',
                 hue_order=season_order,
                 palette={'Winter':'steelblue','Summer':'tomato',
                          'Monsoon':'seagreen','Post-Monsoon':'goldenrod'},
                 plot_kws={'alpha': 0.5, 's': 15},
                 diag_kind='kde')
g.fig.suptitle('Pairplot — Weather Variables by Season', y=1.02, fontsize=13, fontweight='bold')
plt.savefig('../results/pairplot.png', bbox_inches='tight')
plt.show()

---
## 5. Statistical Analysis <a id='5'></a>

Now we aggregate data by **month**, **season**, and **year** to identify trends.

In [ ]:
# ── Monthly averages ─────────────────────────────────────────────────────
monthly = df.groupby(['month', 'month_name']).agg(
    avg_temperature = ('temperature', 'mean'),
    avg_humidity    = ('humidity',    'mean'),
    total_rainfall  = ('rainfall',   'sum'),
    rainy_days      = ('rainfall',   lambda x: (x > 0).sum())
).reset_index().sort_values('month').round(2)

print('📅 Monthly Averages:')
monthly

In [ ]:
# ── Monthly charts ───────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 10))
fig.suptitle('Monthly Weather Patterns', fontsize=15, fontweight='bold')

months_ordered = monthly.sort_values('month')['month_name']

# Temperature
axes[0].bar(months_ordered, monthly.sort_values('month')['avg_temperature'],
            color=plt.cm.coolwarm(np.linspace(0.1, 0.9, 12)), edgecolor='white')
axes[0].set_title('Average Monthly Temperature', fontweight='bold')
axes[0].set_ylabel('Temperature (°C)')
axes[0].axhline(df['temperature'].mean(), color='red', linestyle='--', alpha=0.7, label='Annual Mean')
axes[0].legend()

# Humidity
axes[1].bar(months_ordered, monthly.sort_values('month')['avg_humidity'],
            color='seagreen', alpha=0.75, edgecolor='white')
axes[1].set_title('Average Monthly Humidity', fontweight='bold')
axes[1].set_ylabel('Humidity (%)')

# Rainfall
axes[2].bar(months_ordered, monthly.sort_values('month')['total_rainfall'],
            color='steelblue', alpha=0.8, edgecolor='white')
axes[2].set_title('Total Monthly Rainfall', fontweight='bold')
axes[2].set_ylabel('Rainfall (mm)')
axes[2].set_xlabel('Month')

plt.tight_layout()
plt.savefig('../results/monthly_averages.png')
plt.show()

In [ ]:
# ── Yearly trend ─────────────────────────────────────────────────────────
yearly = df.groupby('year').agg(
    avg_temperature = ('temperature', 'mean'),
    max_temperature = ('temperature', 'max'),
    min_temperature = ('temperature', 'min'),
    total_rainfall  = ('rainfall',   'sum'),
    avg_humidity    = ('humidity',    'mean')
).reset_index().round(2)

print('📈 Year-over-Year Trends:')
display(yearly)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(yearly['year'].astype(str), yearly['avg_temperature'],
        color=['#2196F3','#FF5722','#4CAF50'])
ax1.set_title('Yearly Avg Temperature', fontweight='bold')
ax1.set_ylabel('°C')

ax2.bar(yearly['year'].astype(str), yearly['total_rainfall'],
        color=['#2196F3','#FF5722','#4CAF50'])
ax2.set_title('Yearly Total Rainfall', fontweight='bold')
ax2.set_ylabel('mm')

plt.tight_layout()
plt.savefig('../results/yearly_trend.png')
plt.show()

---
## 6. Seasonal Pattern Analysis <a id='6'></a>

India has 4 meteorological seasons. We compare how weather variables behave across them.

| Season | Months |
|--------|--------|
| Winter | Dec, Jan, Feb |
| Summer | Mar, Apr, May |
| Monsoon | Jun, Jul, Aug, Sep |
| Post-Monsoon | Oct, Nov |

In [ ]:
# ── Seasonal statistics ──────────────────────────────────────────────────
seasonal = df.groupby('season').agg(
    avg_temperature = ('temperature', 'mean'),
    avg_humidity    = ('humidity',    'mean'),
    total_rainfall  = ('rainfall',   'sum'),
    rainy_days      = ('rainfall',   lambda x: (x > 0).sum()),
    total_days      = ('date',        'count')
).reset_index().round(2)

season_order = ['Winter', 'Summer', 'Monsoon', 'Post-Monsoon']
seasonal['season'] = pd.Categorical(seasonal['season'], categories=season_order, ordered=True)
seasonal = seasonal.sort_values('season')

print('🌿 Seasonal Summary:')
display(seasonal)

In [ ]:
# ── Seasonal boxplots ────────────────────────────────────────────────────
season_palette = {'Winter':'steelblue','Summer':'tomato',
                  'Monsoon':'seagreen','Post-Monsoon':'goldenrod'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Weather Variables by Season (Boxplot)', fontsize=14, fontweight='bold')

for ax, col, label in zip(axes,
                           ['temperature','humidity','rainfall'],
                           ['Temperature (°C)','Humidity (%)','Rainfall (mm)']):
    sns.boxplot(data=df, x='season', y=col, order=season_order,
                palette=season_palette, ax=ax, width=0.5)
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel(label)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../results/seasonal_patterns.png')
plt.show()

print('💡 Key Observations:')
print('   - Monsoon has highest humidity AND most rainfall days')
print('   - Summer has highest median temperature')
print('   - Rainfall outliers exist even in dry seasons (occasional rains)')

---
## 7. Forecasting with Moving Averages <a id='7'></a>

### What is a Moving Average?

A **moving average** smooths out short-term fluctuations to reveal long-term trends.

- **SMA (Simple Moving Average):** Average of the last N values — equal weight to all.
  $$SMA_t = \frac{1}{N}\sum_{i=0}^{N-1} x_{t-i}$$

- **EMA (Exponential Moving Average):** More weight to recent values — reacts faster to changes.
  $$EMA_t = \alpha \cdot x_t + (1-\alpha) \cdot EMA_{t-1}$$


In [ ]:
# ── Compute moving averages ──────────────────────────────────────────────
for col in ['temperature', 'humidity', 'rainfall']:
    df[f'{col}_sma7']  = df[col].rolling(window=7,  min_periods=1).mean().round(2)
    df[f'{col}_sma30'] = df[col].rolling(window=30, min_periods=1).mean().round(2)
    df[f'{col}_ema7']  = df[col].ewm(span=7, adjust=False).mean().round(2)

print('✅ Moving averages computed!')
print('   Columns added: _sma7, _sma30, _ema7 for each variable')
df[['date','temperature','temperature_sma7','temperature_sma30','temperature_ema7']].tail(8)

In [ ]:
# ── Temperature trend with SMA ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['date'], df['temperature'],      alpha=0.25, color='steelblue', label='Daily Temp')
ax.plot(df['date'], df['temperature_sma7'], color='orange',  linewidth=1.5,  label='7-Day SMA')
ax.plot(df['date'], df['temperature_sma30'],color='red',     linewidth=2.5,  label='30-Day SMA')
ax.plot(df['date'], df['temperature_ema7'], color='purple',  linewidth=1,    label='7-Day EMA', linestyle='--')

ax.set_title('Temperature Trend with Moving Averages (2021–2023)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.legend()
plt.tight_layout()
plt.savefig('../results/temperature_trend.png')
plt.show()

In [ ]:
# ── Forecast next 7 days ─────────────────────────────────────────────────
def forecast_next_n_days(df, column, n=7):
    """
    Forecast next N days using last 30-day moving average as baseline.
    Adds small realistic variation based on historical std deviation.

    Args:
        df      (pd.DataFrame): Historical DataFrame with 'date' and target column.
        column  (str): Column to forecast.
        n       (int): Number of days to forecast.

    Returns:
        pd.DataFrame: Forecast DataFrame.
    """
    baseline = df[column].tail(30).mean()
    std_dev  = df[column].std()
    last_date = df['date'].max()

    np.random.seed(0)
    variation  = np.random.normal(0, std_dev * 0.1, n)
    predicted  = np.round(baseline + variation, 2)
    fore_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=n)

    return pd.DataFrame({
        'date': fore_dates,
        f'predicted_{column}': predicted
    })

temp_forecast     = forecast_next_n_days(df, 'temperature', 7)
humidity_forecast = forecast_next_n_days(df, 'humidity',    7)

print('🔮 7-Day Temperature Forecast:')
display(temp_forecast)
print('\n🔮 7-Day Humidity Forecast:')
display(humidity_forecast)

In [ ]:
# ── Plot forecast ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('7-Day Weather Forecast using Moving Average', fontsize=13, fontweight='bold')

for ax, col, forecast_df, color in zip(
    axes,
    ['temperature', 'humidity'],
    [temp_forecast, humidity_forecast],
    ['steelblue', 'seagreen']
):
    recent = df.tail(60)
    ax.plot(recent['date'], recent[col],             color=color,    alpha=0.5,  label='Historical (last 60 days)')
    ax.plot(recent['date'], recent[f'{col}_sma7'],   color='orange', linewidth=2, label='7-Day SMA')
    ax.plot(forecast_df['date'], forecast_df[f'predicted_{col}'],
            color='red', marker='o', linestyle='--', linewidth=2,    label='Forecast')
    ax.axvline(x=df['date'].max(), color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
    ax.set_title(f'{col.capitalize()} Forecast', fontweight='bold')
    ax.set_ylabel(col.capitalize())
    ax.legend(fontsize=9)

axes[1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('../results/forecast.png')
plt.show()

---
## 8. Visualization Dashboard <a id='8'></a>

A single combined dashboard summarizing the entire analysis.

In [ ]:
# ── Full dashboard ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.suptitle('Weather Data Analysis Dashboard  |  2021–2023',
             fontsize=17, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Temperature trend — full width
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(df['date'], df['temperature'],       alpha=0.2, color='steelblue', label='Daily')
ax1.plot(df['date'], df['temperature_sma30'], color='red',    linewidth=2,   label='30-Day SMA')
ax1.plot(df['date'], df['temperature_sma7'],  color='orange', linewidth=1.2, label='7-Day SMA')
ax1.set_title('Temperature Trend (2021–2023)')
ax1.set_ylabel('°C')
ax1.legend(fontsize=8)

# 2. Monthly temperature
ax2 = fig.add_subplot(gs[1, :2])
monthly_s = monthly.sort_values('month')
ax2.bar(monthly_s['month_name'], monthly_s['avg_temperature'],
        color=plt.cm.coolwarm(np.linspace(0.1, 0.9, 12)))
ax2.set_title('Avg Monthly Temperature')
ax2.set_ylabel('°C')
ax2.tick_params(axis='x', rotation=45)

# 3. Correlation heatmap
ax3 = fig.add_subplot(gs[1, 2])
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax3, linewidths=0.5, cbar=False, square=True)
ax3.set_title('Correlation')

# 4. Humidity trend
ax4 = fig.add_subplot(gs[2, :2])
ax4.plot(df['date'], df['humidity'],       alpha=0.25, color='seagreen')
ax4.plot(df['date'], df['humidity_sma30'], color='darkgreen', linewidth=2, label='30-Day SMA')
ax4.set_title('Humidity Trend')
ax4.set_ylabel('%')
ax4.legend(fontsize=8)

# 5. Monthly rainfall
ax5 = fig.add_subplot(gs[2, 2])
ax5.bar(monthly_s['month_name'], monthly_s['total_rainfall'],
        color=plt.cm.Blues(np.linspace(0.3, 0.9, 12)))
ax5.set_title('Total Monthly Rainfall')
ax5.set_ylabel('mm')
ax5.tick_params(axis='x', rotation=90)

plt.savefig('../results/dashboard.png', bbox_inches='tight', dpi=120)
plt.show()
print('✅ Dashboard saved to results/dashboard.png')

---
## 9. Conclusion <a id='9'></a>

In [ ]:
# ── Final summary ────────────────────────────────────────────────────────
print('=' * 60)
print('   WEATHER DATA ANALYSIS — PROJECT SUMMARY')
print('=' * 60)
print(f'''
Dataset   : {len(df)} daily records (2021–2023)
Variables : Temperature (°C), Humidity (%), Rainfall (mm)

Key Findings:
  🌡️  Annual avg temperature  : {df['temperature'].mean():.1f}°C
  💧  Annual avg humidity     : {df['humidity'].mean():.1f}%
  🌧️  Total 3-year rainfall   : {df['rainfall'].sum():.0f} mm
  ☔  Total rainy days        : {(df['rainfall'] > 0).sum()} / {len(df)} days

  Hottest month    : {monthly.sort_values('avg_temperature', ascending=False).iloc[0]['month_name']}
  Wettest month    : {monthly.sort_values('total_rainfall',  ascending=False).iloc[0]['month_name']}
  Highest humidity : {monthly.sort_values('avg_humidity',    ascending=False).iloc[0]['month_name']}

Forecasting Model : Simple Moving Average (30-day baseline + EMA smoothing)
Forecast horizon  : 7 days

Charts saved to   : results/ folder
''')
print('=' * 60)
print('✅ Project Complete!')

---

## References

1. pandas Documentation — https://pandas.pydata.org/docs/
2. NumPy Documentation — https://numpy.org/doc/
3. Matplotlib Documentation — https://matplotlib.org/
4. Seaborn Documentation — https://seaborn.pydata.org/
5. OpenWeatherMap API — https://openweathermap.org/api
6. India Meteorological Department — https://mausam.imd.gov.in/

---
*Submitted for Python Programming Subject | B.Tech AI&DS*